# Data loading

In [1]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

Code directory: c:\Users\gfaro\Downloads\Fire-Detection-Data-Mining-Project\Code

Data paths:
  ✓ X_train_smote: c:\Users\gfaro\Downloads\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote.parquet
  ✓ X_train_tomek: c:\Users\gfaro\Downloads\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_tomek.parquet
  ✓ X_train_smote_tomek: c:\Users\gfaro\Downloads\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote_tomek.parquet
  ✓ y_train_smote: c:\Users\gfaro\Downloads\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\smote\y_smote.pkl
  ✓ y_train_tomek: c:\Users\gfaro\Downloads\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\tomek\y_tomek.pkl
  ✓ y_train_smote_tomek: c:\Users\gfaro\Downloads\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_spl

# Clarans predefined

🚀 Key Optimizations Made:

- Sample to 1000 points (from potentially 10,000+) → 10-100x faster
- Reduce to 20 PCA components → 5-25x faster
- Test only 4 k values [2,3,4,5] instead of 9 → 2x faster
- Fewer CLARANS iterations (num_local=2, max_neighbors=30) → 3-5x faster
- Simplified plotting → saves time

In [6]:
# %%
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from pyclustering.cluster.clarans import clarans
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CLARANS CLUSTERING - FIXED k=2 WITH CHECKPOINTS")
print("=" * 80)

# ========================================
# CONFIGURATION
# ========================================

MAX_SAMPLES = 500       # Reduced from 1000 for speed
MAX_FEATURES = 15       # Reduced from 20 for speed
K_CLUSTERS = 2
NUM_LOCAL = 2
MAX_NEIGHBORS = 20      # Reduced from 30 for speed

# Create output folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FOLDER = f"clarans_results_{timestamp}"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"📁 Output folder: {OUTPUT_FOLDER}")

# Checkpoint file
CHECKPOINT_FILE = os.path.join(OUTPUT_FOLDER, "checkpoint.pkl")
RESULTS_CSV = os.path.join(OUTPUT_FOLDER, "results_summary.csv")

# ========================================
# CHECKPOINT FUNCTIONS
# ========================================

def save_checkpoint(all_results, current_method=None):
    """Save checkpoint with current results"""
    checkpoint_data = {
        'all_results': all_results,
        'current_method': current_method,
        'timestamp': datetime.now().isoformat()
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(checkpoint_data, f)
    print(f"💾 Checkpoint saved: {CHECKPOINT_FILE}")

def load_checkpoint():
    """Load checkpoint if exists"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            checkpoint_data = pickle.load(f)
        print(f"✅ Checkpoint loaded from: {CHECKPOINT_FILE}")
        print(f"   Last saved: {checkpoint_data['timestamp']}")
        print(f"   Methods completed: {len(checkpoint_data['all_results'])}")
        return checkpoint_data['all_results'], checkpoint_data['current_method']
    return [], None

def save_incremental_results(all_results):
    """Save results to CSV incrementally"""
    if not all_results:
        return
    
    comparison_data = []
    for r in all_results:
        m = r['metrics']
        row = {
            'Method': r['method'],
            'Silhouette': m['silhouette'],
            'Calinski-H': m['calinski'],
            'Davies-B': m['davies_bouldin'],
            'ARI': m['ari'],
            'NMI': m['nmi'],
            'Time (s)': m['execution_time'],
            'Cluster_0_size': r['cluster_sizes'].get(0, 0),
            'Cluster_1_size': r['cluster_sizes'].get(1, 0)
        }
        comparison_data.append(row)
    
    df = pd.DataFrame(comparison_data)
    df.to_csv(RESULTS_CSV, index=False)
    print(f"💾 Results saved: {RESULTS_CSV}")

# ========================================
# UTILITY FUNCTIONS
# ========================================

def sample_dataset(X, y, max_samples):
    """Randomly sample dataset if too large"""
    if len(X) <= max_samples:
        return X, y
    
    print(f"⚡ Sampling {max_samples} from {len(X)} samples...")
    indices = np.random.choice(len(X), max_samples, replace=False)
    return X[indices], y.iloc[indices] if hasattr(y, 'iloc') else y[indices]

def reduce_dimensions(X, max_components):
    """Reduce dimensionality with PCA"""
    if X.shape[1] <= max_components:
        return X, None
    
    print(f"⚡ Reducing {X.shape[1]} features to {max_components} with PCA...")
    pca = PCA(n_components=max_components)
    X_reduced = pca.fit_transform(X)
    variance = pca.explained_variance_ratio_.sum()
    print(f"   Variance retained: {variance:.2%}")
    return X_reduced, pca

def run_clarans(X, k, num_local, max_neighbors):
    """Run CLARANS clustering"""
    data_points = X.tolist()
    
    clarans_instance = clarans(data_points, k, num_local, max_neighbors)
    clarans_instance.process()
    
    clusters = clarans_instance.get_clusters()
    medoids = clarans_instance.get_medoids()
    
    # Convert to labels
    labels = np.zeros(len(X), dtype=int)
    for cluster_id, cluster in enumerate(clusters):
        for point_idx in cluster:
            labels[point_idx] = cluster_id
    
    # Calculate inertia
    inertia = 0
    for cluster_id, cluster in enumerate(clusters):
        medoid = data_points[medoids[cluster_id]]
        for point_idx in cluster:
            point = data_points[point_idx]
            dist = sum((a - b) ** 2 for a, b in zip(point, medoid))
            inertia += dist
    
    return labels, medoids, inertia

def calculate_metrics(X, labels):
    """Calculate clustering metrics"""
    if len(np.unique(labels)) < 2:
        return {'silhouette': 0, 'calinski': 0, 'davies_bouldin': float('inf')}
    
    try:
        sil = silhouette_score(X, labels)
        cal = calinski_harabasz_score(X, labels)
        db = davies_bouldin_score(X, labels)
    except:
        return {'silhouette': 0, 'calinski': 0, 'davies_bouldin': float('inf')}
    
    return {'silhouette': sil, 'calinski': cal, 'davies_bouldin': db}

def plot_clusters(X, labels, medoids, method_name):
    """Plot clusters in 2D"""
    if X.shape[1] > 2:
        pca = PCA(n_components=2)
        X_2d = pca.fit_transform(X)
        medoids_2d = X_2d[medoids]
        var_exp = pca.explained_variance_ratio_
        title_suffix = f" (PCA: {var_exp[0]:.1%}+{var_exp[1]:.1%} var)"
    else:
        X_2d = X
        medoids_2d = X[medoids]
        title_suffix = ""
    
    plt.figure(figsize=(10, 8))
    
    # Plot points
    unique_labels = np.unique(labels)
    colors = ['#1f77b4', '#ff7f0e']
    
    for label, color in zip(unique_labels, colors):
        mask = labels == label
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                   c=color, label=f'Cluster {label}', 
                   alpha=0.6, s=50, edgecolors='w', linewidth=0.5)
    
    # Plot medoids
    plt.scatter(medoids_2d[:, 0], medoids_2d[:, 1], 
               c='red', marker='X', s=300, 
               label='Medoids', edgecolors='black', linewidth=2)
    
    plt.title(f'CLARANS Clusters - {method_name}{title_suffix}', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    filename = os.path.join(OUTPUT_FOLDER, 
                           f"clusters_{method_name.lower().replace(' ', '_')}.png")
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"  💾 Saved: {filename}")
    return filename

def plot_confusion(y_true, y_pred, method_name):
    """Plot confusion matrix"""
    confusion = pd.crosstab(
        pd.Series(y_pred, name='Cluster'),
        pd.Series(y_true, name='True Class')
    )
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Count'})
    plt.title(f'Confusion Matrix - {method_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    filename = os.path.join(OUTPUT_FOLDER,
                           f"confusion_{method_name.lower().replace(' ', '_')}.png")
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"  💾 Saved: {filename}")
    return filename

# ========================================
# MAIN ANALYSIS
# ========================================

if 'data' not in globals():
    print("❌ Data not loaded. Please load your data first.")
    exit()

METHODS = [
    ('SMOTE', 'X_train_smote', 'y_train_smote'),
    ('Tomek Links', 'X_train_tomek', 'y_train_tomek'),
    ('SMOTE + Tomek Links', 'X_train_smote_tomek', 'y_train_smote_tomek')
]

# Try to load checkpoint
all_results, last_method = load_checkpoint()
completed_methods = [r['method'] for r in all_results]

print(f"\n🔄 Progress: {len(completed_methods)}/{len(METHODS)} methods completed")
if completed_methods:
    print(f"   Already completed: {', '.join(completed_methods)}")

for method_name, X_train_key, y_train_key in METHODS:
    
    # Skip if already completed
    if method_name in completed_methods:
        print(f"\n⏭️  Skipping {method_name} (already completed)")
        continue
    
    print(f"\n{'=' * 80}")
    print(f"🚀 Processing: {method_name} ({len(all_results)+1}/{len(METHODS)})")
    print(f"{'=' * 80}")
    
    # Check data availability
    if X_train_key not in data or y_train_key not in data:
        print(f"⚠️ {method_name} data not available, skipping...")
        continue
    
    try:
        # Load data
        X_train = data[X_train_key].values
        y_train = data[y_train_key]
        
        print(f"📊 Original shape: {X_train.shape}")
        print(f"🎯 Class distribution: {Counter(y_train)}")
        
        # Sample and reduce
        X_sample, y_sample = sample_dataset(X_train, y_train, MAX_SAMPLES)
        X_reduced, pca = reduce_dimensions(X_sample, MAX_FEATURES)
        print(f"📊 Working shape: {X_reduced.shape}")
        
        # Run CLARANS with k=2
        print(f"\n🎯 Running CLARANS with k={K_CLUSTERS}...")
        start = time.time()
        
        labels, medoids, inertia = run_clarans(
            X_reduced, K_CLUSTERS, NUM_LOCAL, MAX_NEIGHBORS
        )
        
        exec_time = time.time() - start
        print(f"✅ Completed in {exec_time:.2f}s")
        
        # Calculate metrics
        metrics = calculate_metrics(X_reduced, labels)
        metrics['execution_time'] = exec_time
        metrics['inertia'] = inertia
        
        # Cluster sizes
        cluster_sizes = Counter(labels)
        print(f"📊 Cluster sizes: {dict(cluster_sizes)}")
        
        # Compare with true labels
        ari = adjusted_rand_score(y_sample, labels)
        nmi = normalized_mutual_info_score(y_sample, labels)
        metrics['ari'] = ari
        metrics['nmi'] = nmi
        
        print(f"\n📊 Clustering Metrics:")
        print(f"  • Silhouette: {metrics['silhouette']:.4f}")
        print(f"  • Calinski-Harabasz: {metrics['calinski']:.2f}")
        print(f"  • Davies-Bouldin: {metrics['davies_bouldin']:.4f}")
        print(f"  • Inertia: {inertia:.2f}")
        print(f"\n🔍 Comparison with True Labels:")
        print(f"  • ARI: {ari:.4f}")
        print(f"  • NMI: {nmi:.4f}")
        
        # Visualizations
        print(f"\n📊 Generating plots...")
        cluster_plot = plot_clusters(X_reduced, labels, medoids, method_name)
        confusion_plot = plot_confusion(y_sample, labels, method_name)
        
        # Store results
        result = {
            'method': method_name,
            'metrics': metrics,
            'cluster_sizes': dict(cluster_sizes),
            'plots': [cluster_plot, confusion_plot]
        }
        
        all_results.append(result)
        
        # Save checkpoint and incremental results
        print(f"\n💾 Saving checkpoint...")
        save_checkpoint(all_results, method_name)
        save_incremental_results(all_results)
        
        print(f"\n✅ {method_name} completed successfully!")
        
    except Exception as e:
        print(f"\n❌ ERROR processing {method_name}: {str(e)}")
        print(f"   Saving checkpoint before continuing...")
        save_checkpoint(all_results, method_name)
        continue

# ========================================
# FINAL COMPARISON
# ========================================

if all_results:
    print(f"\n{'=' * 80}")
    print("📊 FINAL COMPARISON (k=2)")
    print(f"{'=' * 80}\n")
    
    # Create comparison table
    comparison_data = []
    for r in all_results:
        m = r['metrics']
        row = {
            'Method': r['method'],
            'Silhouette': m['silhouette'],
            'Calinski-H': m['calinski'],
            'Davies-B': m['davies_bouldin'],
            'ARI': m['ari'],
            'NMI': m['nmi'],
            'Time (s)': m['execution_time']
        }
        comparison_data.append(row)
    
    df = pd.DataFrame(comparison_data)
    print(df.to_string(index=False))
    
    # Save final comparison
    final_csv = os.path.join(OUTPUT_FOLDER, "final_comparison.csv")
    df.to_csv(final_csv, index=False)
    print(f"\n💾 Saved: {final_csv}")
    
    # Find best method based on Silhouette
    best_idx = df['Silhouette'].idxmax()
    print(f"\n🏆 BEST METHOD: {df.loc[best_idx, 'Method']}")
    print(f"   Silhouette: {df.loc[best_idx, 'Silhouette']:.4f}")
    print(f"   ARI: {df.loc[best_idx, 'ARI']:.4f}")
    
    # Comparison plots
    print(f"\n📊 Creating final comparison plot...")
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    methods = df['Method'].values
    
    metrics_to_plot = [
        ('Silhouette Score', 'Silhouette', 'higher'),
        ('Calinski-Harabasz', 'Calinski-H', 'higher'),
        ('Davies-Bouldin', 'Davies-B', 'lower'),
        ('Adjusted Rand Index', 'ARI', 'higher'),
        ('Normalized Mutual Info', 'NMI', 'higher'),
        ('Execution Time (s)', 'Time (s)', 'lower')
    ]
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    for idx, (title, col, better) in enumerate(metrics_to_plot):
        ax = axes[idx // 3, idx % 3]
        values = df[col].values
        
        bars = ax.bar(methods, values, color=colors, edgecolor='black', alpha=0.7)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Highlight best
        if better == 'higher':
            best = np.argmax(values)
        else:
            best = np.argmin(values)
        bars[best].set_color('green')
        bars[best].set_alpha(0.9)
        
        # Add values on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}' if height < 100 else f'{height:.1f}',
                   ha='center', va='bottom', fontsize=9)
    
    plt.suptitle(f'CLARANS Method Comparison (k=2)', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    plot_file = os.path.join(OUTPUT_FOLDER, "final_comparison.png")
    plt.savefig(plot_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"💾 Saved: {plot_file}")
    
    # Summary of all files
    print(f"\n{'=' * 80}")
    print(f"📁 ALL FILES IN: {OUTPUT_FOLDER}")
    print(f"{'=' * 80}")
    
    print(f"\n📄 Summary files:")
    print(f"  • {RESULTS_CSV}")
    print(f"  • {final_csv}")
    print(f"  • {plot_file}")
    print(f"  • {CHECKPOINT_FILE}")
    
    print(f"\n🖼️  Method-specific plots:")
    for r in all_results:
        print(f"\n  {r['method']}:")
        for plot in r['plots']:
            print(f"    • {os.path.basename(plot)}")
    
    print(f"\n💡 Tip: If interrupted, re-run the script to resume from checkpoint!")

else:
    print("\n⚠️  No results available")

print(f"\n{'=' * 80}")
print("✅ ANALYSIS COMPLETE!")
print(f"{'=' * 80}")

CLARANS CLUSTERING - FIXED k=2 WITH CHECKPOINTS
📁 Output folder: clarans_results_20260102_225230

🔄 Progress: 0/3 methods completed

🚀 Processing: SMOTE (1/3)
📊 Original shape: (30570, 64)
🎯 Class distribution: Counter({0: 15285, 1: 15285})
⚡ Sampling 500 from 30570 samples...
⚡ Reducing 64 features to 15 with PCA...
   Variance retained: 100.00%
📊 Working shape: (500, 15)

🎯 Running CLARANS with k=2...


KeyboardInterrupt: 

# Bayesian Search

In [ ]:
# %%
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from pyclustering.cluster.clarans import clarans
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time
from datetime import datetime
import warnings
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
from skopt import gp_minimize
from skopt.plots import plot_convergence, plot_objective
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("BAYESIAN OPTIMIZATION POUR CLARANS CLUSTERING")
print("=" * 80)

# ========================================
# PARTIE 1: CLASSE WRAPPER POUR SKOPT
# ========================================

class ClaransWrapper:
    """Wrapper pour CLARANS compatible avec scikit-learn et skopt"""
    
    def __init__(self, num_clusters=3, num_local=2, max_neighbors=50):
        self.num_clusters = num_clusters
        self.num_local = num_local
        self.max_neighbors = max_neighbors
        self.labels_ = None
        self.clusters_ = None
        self.medoids_ = None
        self.inertia_ = None
        
    def fit(self, X, y=None):
        """Entraîne le modèle CLARANS"""
        # Convertir en liste pour pyclustering
        data_points = X.tolist()
        
        # Créer et exécuter CLARANS
        clarans_instance = clarans(
            data_points, 
            self.num_clusters, 
            self.num_local,
            self.max_neighbors
        )
        
        clarans_instance.process()
        
        # Stocker les résultats
        self.clusters_ = clarans_instance.get_clusters()
        self.medoids_ = clarans_instance.get_medoids()
        
        # Convertir en labels
        self.labels_ = np.zeros(len(X), dtype=int)
        for cluster_id, cluster in enumerate(self.clusters_):
            for point_idx in cluster:
                self.labels_[point_idx] = cluster_id
        
        # Calculer l'inertie
        self.inertia_ = self._calculate_inertia(X, self.clusters_, self.medoids_)
        
        return self
    
    def _calculate_inertia(self, X, clusters, medoids):
        """Calcule l'inertie (somme des distances au carré)"""
        inertia = 0
        data_points = X.tolist()
        
        for cluster_id, cluster in enumerate(clusters):
            medoid = data_points[medoids[cluster_id]]
            for point_idx in cluster:
                point = data_points[point_idx]
                # Distance euclidienne au carré
                dist = sum((a - b) ** 2 for a, b in zip(point, medoid))
                inertia += dist
        
        return inertia
    
    def predict(self, X):
        """Prédit les clusters pour de nouvelles données (méthode des plus proches médoides)"""
        if self.medoids_ is None:
            raise ValueError("Le modèle doit être entraîné avant la prédiction")
        
        # Récupérer les médoides d'origine
        data_points = X.tolist()
        medoids_points = [data_points[i] for i in self.medoids_]
        
        # Assigner chaque point au médioïde le plus proche
        labels = []
        for point in data_points:
            distances = [sum((a - b) ** 2 for a, b in zip(point, medoid)) 
                        for medoid in medoids_points]
            labels.append(np.argmin(distances))
        
        return np.array(labels)

# ========================================
# PARTIE 2: FONCTIONS D'ÉVALUATION ET VISUALISATION
# ========================================

def calculate_clustering_metrics(X, labels, method_name):
    """Calcule toutes les métriques d'évaluation du clustering"""
    if len(np.unique(labels)) < 2:
        return {
            'silhouette': 0,
            'calinski_harabasz': 0,
            'davies_bouldin': float('inf'),
            'n_clusters': len(np.unique(labels)),
            'cluster_sizes': Counter(labels)
        }
    
    try:
        silhouette = silhouette_score(X, labels)
    except:
        silhouette = 0
    
    try:
        calinski = calinski_harabasz_score(X, labels)
    except:
        calinski = 0
    
    try:
        davies = davies_bouldin_score(X, labels)
    except:
        davies = float('inf')
    
    return {
        'silhouette': silhouette,
        'calinski_harabasz': calinski,
        'davies_bouldin': davies,
        'n_clusters': len(np.unique(labels)),
        'cluster_sizes': Counter(labels),
        'method': method_name
    }

def objective_function(params, X, scoring='silhouette'):
    """Fonction objectif pour l'optimisation bayésienne"""
    num_clusters, num_local, max_neighbors = params
    
    try:
        # Créer et entraîner CLARANS
        model = ClaransWrapper(
            num_clusters=int(num_clusters),
            num_local=int(num_local),
            max_neighbors=int(max_neighbors)
        )
        
        model.fit(X)
        labels = model.labels_
        
        # Calculer le score selon la métrique choisie
        if scoring == 'silhouette':
            if len(np.unique(labels)) < 2:
                return 0
            score = silhouette_score(X, labels)
        elif scoring == 'calinski_harabasz':
            if len(np.unique(labels)) < 2:
                return 0
            score = calinski_harabasz_score(X, labels)
        elif scoring == 'davies_bouldin':
            if len(np.unique(labels)) < 2:
                return float('inf')
            score = -davies_bouldin_score(X, labels)  # Négatif car on veut minimiser
        elif scoring == 'inertia':
            score = -model.inertia_  # Négatif car on veut minimiser l'inertie
        else:
            raise ValueError(f"Métrique non supportée: {scoring}")
        
        return score
        
    except Exception as e:
        print(f"Erreur avec params {params}: {e}")
        if scoring == 'davies_bouldin':
            return float('inf')
        return 0

def bayesian_optimization_clarans(X, method_name, timestamp, 
                                  n_calls=30, scoring='silhouette'):
    """Optimisation bayésienne pour CLARANS"""
    print(f"\n🎯 OPTIMISATION BAYÉSIENNE POUR CLARANS")
    print(f"   Méthode: {method_name}")
    print(f"   Métrique: {scoring}")
    print(f"   Nombre d'appels: {n_calls}")
    
    start_time = time.time()
    
    # Définir l'espace de recherche
    search_space = [
        Integer(2, 15, name='num_clusters'),      # Nombre de clusters
        Integer(2, 10, name='num_local'),         # Nombre de minimas locaux
        Integer(10, 200, name='max_neighbors')    # Nombre maximum de voisins
    ]
    
    # Fonction objectif adaptée pour skopt
    @use_named_args(search_space)
    def objective_skopt(num_clusters, num_local, max_neighbors):
        return -objective_function(  # Négatif car gp_minimize minimise
            [num_clusters, num_local, max_neighbors], 
            X, 
            scoring
        )
    
    # Exécuter l'optimisation bayésienne
    print("  🔍 Lancement de l'optimisation bayésienne...")
    result = gp_minimize(
        func=objective_skopt,
        dimensions=search_space,
        n_calls=n_calls,
        n_random_starts=10,
        acq_func='EI',  # Expected Improvement
        random_state=42,
        verbose=True
    )
    
    opt_time = time.time() - start_time
    
    print(f"\n✅ Optimisation terminée en {opt_time:.2f} secondes")
    print(f"\n🎯 MEILLEURS PARAMÈTRES TROUVÉS:")
    print(f"   Nombre de clusters: {result.x[0]}")
    print(f"   Nombre de minimas locaux: {result.x[1]}")
    print(f"   Nombre maximum de voisins: {result.x[2]}")
    print(f"   Meilleur score ({scoring}): {-result.fun:.4f}")  # Négatif car on a minimisé -score
    
    # Entraîner le modèle final avec les meilleurs paramètres
    print("\n🧠 Entraînement du modèle final avec les meilleurs paramètres...")
    final_model = ClaransWrapper(
        num_clusters=int(result.x[0]),
        num_local=int(result.x[1]),
        max_neighbors=int(result.x[2])
    )
    
    final_model.fit(X)
    
    # Calculer toutes les métriques
    all_metrics = calculate_clustering_metrics(X, final_model.labels_, method_name)
    all_metrics['optimization_time'] = opt_time
    all_metrics['best_params'] = {
        'num_clusters': int(result.x[0]),
        'num_local': int(result.x[1]),
        'max_neighbors': int(result.x[2])
    }
    all_metrics['best_score'] = -result.fun
    all_metrics['scoring_metric'] = scoring
    
    # Visualisations
    plots = {}
    
    # 1. Graphique de convergence
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    try:
        plot_convergence(result, ax=axes[0, 0])
        axes[0, 0].set_title('Convergence de l\'optimisation', fontsize=12, fontweight='bold')
    except:
        axes[0, 0].text(0.5, 0.5, 'Graphique de convergence\nnon disponible', 
                       ha='center', va='center', fontsize=12)
    
    # 2. Distribution des scores
    if hasattr(result, 'func_vals'):
        axes[0, 1].hist(-np.array(result.func_vals), bins=20, 
                       color='skyblue', edgecolor='black', alpha=0.7)
        axes[0, 1].axvline(x=-result.fun, color='r', linestyle='--', 
                          linewidth=2, label=f'Meilleur: {-result.fun:.4f}')
        axes[0, 1].set_xlabel(f'Score ({scoring})', fontsize=11)
        axes[0, 1].set_ylabel('Fréquence', fontsize=11)
        axes[0, 1].set_title('Distribution des scores', fontsize=12, fontweight='bold')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Évolution des paramètres
    if hasattr(result, 'x_iters'):
        iterations = range(1, len(result.x_iters) + 1)
        
        # Nombre de clusters
        axes[1, 0].plot(iterations, [x[0] for x in result.x_iters], 'b-o', 
                       linewidth=2, markersize=4, label='num_clusters')
        axes[1, 0].axhline(y=result.x[0], color='r', linestyle='--', 
                          label=f'Optimal: {result.x[0]}')
        axes[1, 0].set_xlabel('Itération', fontsize=11)
        axes[1, 0].set_ylabel('Nombre de clusters', fontsize=11)
        axes[1, 0].set_title('Évolution du nombre de clusters', fontsize=12, fontweight='bold')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Nombre de voisins
        axes[1, 1].plot(iterations, [x[2] for x in result.x_iters], 'g-s', 
                       linewidth=2, markersize=4, label='max_neighbors')
        axes[1, 1].axhline(y=result.x[2], color='r', linestyle='--', 
                          label=f'Optimal: {result.x[2]}')
        axes[1, 1].set_xlabel('Itération', fontsize=11)
        axes[1, 1].set_ylabel('Nombre de voisins', fontsize=11)
        axes[1, 1].set_title('Évolution du nombre de voisins', fontsize=12, fontweight='bold')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(f'Optimisation Bayésienne CLARANS - {method_name}', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    convergence_plot = f"clarans_bayesian_convergence_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
    plt.savefig(convergence_plot, dpi=300, bbox_inches='tight')
    plt.close()
    plots['convergence'] = convergence_plot
    print(f"💾 Graphique de convergence sauvegardé: {convergence_plot}")
    
    # 4. Visualisation des clusters (2D PCA)
    from sklearn.decomposition import PCA
    
    if X.shape[1] > 2:
        pca = PCA(n_components=2)
        X_2d = pca.fit_transform(X)
        variance_exp = pca.explained_variance_ratio_
        title_suffix = f" (PCA: {variance_exp[0]:.1%}+{variance_exp[1]:.1%} variance)"
    else:
        X_2d = X
        title_suffix = ""
    
    # Obtenir les médoides en 2D
    medoids_2d = X_2d[final_model.medoids_] if final_model.medoids_ is not None else None
    
    plt.figure(figsize=(10, 8))
    unique_labels = np.unique(final_model.labels_)
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colors):
        mask = final_model.labels_ == label
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                   c=[color], label=f'Cluster {label}', 
                   alpha=0.6, s=50, edgecolors='w', linewidth=0.5)
    
    if medoids_2d is not None:
        plt.scatter(medoids_2d[:, 0], medoids_2d[:, 1], 
                   c='red', marker='X', s=300, 
                   label='Médoides', edgecolors='black', linewidth=2)
    
    plt.title(f'Clusters CLARANS (optimisés) - {method_name}{title_suffix}', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Component 1', fontsize=12)
    plt.ylabel('Component 2', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    clusters_plot = f"clarans_bayesian_clusters_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
    plt.savefig(clusters_plot, dpi=300, bbox_inches='tight')
    plt.close()
    plots['clusters'] = clusters_plot
    print(f"💾 Visualisation des clusters sauvegardée: {clusters_plot}")
    
    # 5. Analyse des résultats par cluster
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Taille des clusters
    cluster_sizes = [np.sum(final_model.labels_ == i) for i in range(len(np.unique(final_model.labels_)))]
    bars = axes[0].bar(range(len(cluster_sizes)), cluster_sizes, 
                      color='lightcoral', edgecolor='black')
    axes[0].set_xlabel('Cluster ID', fontsize=12)
    axes[0].set_ylabel('Nombre de points', fontsize=12)
    axes[0].set_title('Distribution des tailles de clusters', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    for i, bar in enumerate(bars):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    str(int(height)), ha='center', va='bottom', fontsize=10)
    
    # Métriques de qualité
    metrics_names = ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']
    metrics_values = [
        all_metrics['silhouette'],
        all_metrics['calinski_harabasz'] / 1000 if all_metrics['calinski_harabasz'] > 1000 else all_metrics['calinski_harabasz'],
        1 / all_metrics['davies_bouldin'] if all_metrics['davies_bouldin'] > 0 else 0
    ]
    
    colors_metrics = ['green' if m > 0.5 else 'orange' if m > 0.3 else 'red' 
                     for m in [all_metrics['silhouette'], 
                              all_metrics['calinski_harabasz'] / max(1000, all_metrics['calinski_harabasz']),
                              1 - min(1, all_metrics['davies_bouldin'] / 5)]]
    
    bars_metrics = axes[1].bar(metrics_names, metrics_values, 
                              color=colors_metrics, edgecolor='black', alpha=0.7)
    axes[1].set_ylabel('Score (normalisé)', fontsize=12)
    axes[1].set_title('Métriques de qualité du clustering', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    for bar, value, orig_value in zip(bars_metrics, metrics_values, 
                                      [all_metrics['silhouette'], 
                                       all_metrics['calinski_harabasz'],
                                       all_metrics['davies_bouldin']]):
        height = bar.get_height()
        if metrics_names[bars_metrics.tolist().index(bar)] == 'Davies-Bouldin':
            axes[1].text(bar.get_x() + bar.get_width()/2., height,
                        f'{orig_value:.3f}', ha='center', va='bottom', fontsize=10)
        else:
            axes[1].text(bar.get_x() + bar.get_width()/2., height,
                        f'{orig_value:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.suptitle(f'Analyse des résultats - {method_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    analysis_plot = f"clarans_bayesian_analysis_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
    plt.savefig(analysis_plot, dpi=300, bbox_inches='tight')
    plt.close()
    plots['analysis'] = analysis_plot
    print(f"💾 Analyse des résultats sauvegardée: {analysis_plot}")
    
    return {
        'model': final_model,
        'metrics': all_metrics,
        'optimization_result': result,
        'plots': plots
    }

def compare_scoring_metrics(X, method_name, timestamp):
    """Compare différentes métriques d'évaluation pour l'optimisation"""
    print(f"\n📊 COMPARAISON DES MÉTRIQUES D'ÉVALUATION")
    print(f"   Méthode: {method_name}")
    
    scoring_metrics = ['silhouette', 'calinski_harabasz', 'davies_bouldin', 'inertia']
    results_by_metric = {}
    
    for scoring in scoring_metrics:
        print(f"\n  🔍 Optimisation avec métrique: {scoring}")
        try:
            result = bayesian_optimization_clarans(
                X, 
                f"{method_name} ({scoring})", 
                timestamp,
                n_calls=15,  # Moins d'appels pour la comparaison
                scoring=scoring
            )
            results_by_metric[scoring] = result
        except Exception as e:
            print(f"    ❌ Erreur avec {scoring}: {e}")
            continue
    
    # Comparer les résultats
    if results_by_metric:
        print(f"\n📋 COMPARAISON DES MÉTRIQUES:")
        print("-" * 80)
        print(f"{'Métrique':<20} {'Silhouette':<12} {'Calinski':<12} {'Davies':<12} {'Clusters':<10}")
        print("-" * 80)
        
        comparison_data = []
        for scoring, result in results_by_metric.items():
            metrics = result['metrics']
            comparison_data.append({
                'scoring': scoring,
                'silhouette': metrics['silhouette'],
                'calinski_harabasz': metrics['calinski_harabasz'],
                'davies_bouldin': metrics['davies_bouldin'],
                'n_clusters': metrics['n_clusters'],
                'best_score': metrics['best_score']
            })
            
            print(f"{scoring:<20} "
                  f"{metrics['silhouette']:<12.4f} "
                  f"{metrics['calinski_harabasz']:<12.1f} "
                  f"{metrics['davies_bouldin']:<12.4f} "
                  f"{metrics['n_clusters']:<10}")
        
        print("-" * 80)
        
        # Trouver la meilleure métrique selon le score Silhouette
        best_metric = max(comparison_data, key=lambda x: x['silhouette'])
        print(f"\n🏆 MEILLEURE MÉTRIQUE: {best_metric['scoring']}")
        print(f"   Silhouette Score: {best_metric['silhouette']:.4f}")
        print(f"   Nombre de clusters: {best_metric['n_clusters']}")
        
        # Graphique de comparaison
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        metrics_list = [d['scoring'] for d in comparison_data]
        silhouette_scores = [d['silhouette'] for d in comparison_data]
        calinski_scores = [d['calinski_harabasz'] for d in comparison_data]
        davies_scores = [d['davies_bouldin'] for d in comparison_data]
        n_clusters = [d['n_clusters'] for d in comparison_data]
        
        # Normaliser Calinski-Harabasz pour l'affichage
        calinski_norm = [c / max(calinski_scores) if max(calinski_scores) > 0 else 0 
                        for c in calinski_scores]
        
        # Silhouette Scores
        bars1 = axes[0].bar(metrics_list, silhouette_scores, 
                           color=['green' if s > 0.5 else 'orange' if s > 0.3 else 'red' 
                                 for s in silhouette_scores],
                           edgecolor='black')
        axes[0].set_title('Silhouette Scores', fontsize=13, fontweight='bold')
        axes[0].set_ylabel('Score', fontsize=11)
        axes[0].set_ylim([0, 1])
        axes[0].grid(True, alpha=0.3, axis='y')
        
        for bar in bars1:
            height = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2., height,
                        f'{height:.3f}', ha='center', va='bottom', fontsize=10)
        
        # Calinski-Harabasz (normalisé)
        bars2 = axes[1].bar(metrics_list, calinski_norm, 
                           color='skyblue', edgecolor='black')
        axes[1].set_title('Calinski-Harabasz (normalisé)', fontsize=13, fontweight='bold')
        axes[1].set_ylabel('Score normalisé', fontsize=11)
        axes[1].set_ylim([0, 1])
        axes[1].grid(True, alpha=0.3, axis='y')
        
        for bar, orig in zip(bars2, calinski_scores):
            height = bar.get_height()
            axes[1].text(bar.get_x() + bar.get_width()/2., height,
                        f'{orig:.1f}', ha='center', va='bottom', fontsize=10)
        
        # Nombre de clusters
        bars3 = axes[2].bar(metrics_list, n_clusters, 
                           color='lightcoral', edgecolor='black')
        axes[2].set_title('Nombre de clusters trouvés', fontsize=13, fontweight='bold')
        axes[2].set_ylabel('Nombre de clusters', fontsize=11)
        axes[2].grid(True, alpha=0.3, axis='y')
        
        for bar in bars3:
            height = bar.get_height()
            axes[2].text(bar.get_x() + bar.get_width()/2., height,
                        str(int(height)), ha='center', va='bottom', fontsize=10)
        
        plt.suptitle(f'Comparaison des métriques d\'optimisation - {method_name}', 
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        comparison_plot = f"clarans_metrics_comparison_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
        plt.savefig(comparison_plot, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"💾 Graphique de comparaison sauvegardé: {comparison_plot}")
        
        return results_by_metric, best_metric['scoring']
    
    return None, None

# ========================================
# PARTIE 3: ÉVALUATION SUR TOUTES LES MÉTHODES
# ========================================

# Vérifier que les données sont chargées
if 'data' not in globals():
    print("❌ Les données ne sont pas chargées. Veuillez exécuter le code de chargement d'abord.")
    exit()

print(f"✅ Données chargées: {len(data)} datasets")

# Définir les méthodes disponibles
METHODS = [
    ('SMOTE', 'X_train_smote', 'y_train_smote'),
    ('Tomek Links', 'X_train_tomek', 'y_train_tomek'),
    ('SMOTE + Tomek Links', 'X_train_smote_tomek', 'y_train_smote_tomek')
]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
all_results = []

# ========================================
# PARTIE 4: BOUCLE SUR TOUTES LES MÉTHODES
# ========================================

for method_name, X_train_key, y_train_key in METHODS:
    print(f"\n" + "=" * 80)
    print(f"🚀 OPTIMISATION BAYÉSIENNE CLARANS - {method_name}")
    print("=" * 80)
    
    # Vérifier si les données existent
    if X_train_key not in data or y_train_key not in data:
        print(f"⚠️  Données {method_name} non disponibles, skip...")
        continue
    
    # Préparer les données
    X_train = data[X_train_key].values
    y_train = data[y_train_key]
    
    print(f"📊 Données: {X_train.shape[0]} échantillons, {X_train.shape[1]} features")
    print(f"🎯 Distribution des classes originales: {Counter(y_train)}")
    
    # Normalisation des données
    print("\n🔄 Normalisation des données...")
    scaler = StandardScaler()
    X_normalized = scaler.fit_transform(X_train)
    print("✅ Normalisation terminée")
    
    # Réduction de dimension si trop de features (pour accélérer)
    if X_normalized.shape[1] > 100:
        print(f"⚠️  Trop de features ({X_normalized.shape[1]}), application de PCA...")
        from sklearn.decomposition import PCA
        pca = PCA(n_components=min(100, X_normalized.shape[0]))
        X_normalized = pca.fit_transform(X_normalized)
        print(f"✅ Réduction à {X_normalized.shape[1]} composantes principales")
        print(f"   Variance expliquée: {pca.explained_variance_ratio_.sum():.2%}")
    
    # Étape 1: Comparer différentes métriques d'évaluation
    print("\n" + "-" * 60)
    print("ÉTAPE 1: COMPARAISON DES MÉTRIQUES D'ÉVALUATION")
    print("-" * 60)
    
    results_by_metric, best_scoring = compare_scoring_metrics(
        X_normalized, method_name, timestamp
    )
    
    # Étape 2: Optimisation complète avec la meilleure métrique
    print("\n" + "-" * 60)
    print(f"ÉTAPE 2: OPTIMISATION COMPLÈTE AVEC MÉTRIQUE '{best_scoring}'")
    print("-" * 60)
    
    final_result = bayesian_optimization_clarans(
        X_normalized,
        method_name,
        timestamp,
        n_calls=30,  # Nombre d'appels pour l'optimisation finale
        scoring=best_scoring if best_scoring else 'silhouette'
    )
    
    # Étape 3: Comparaison avec les labels originaux
    print("\n" + "-" * 60)
    print("ÉTAPE 3: COMPARAISON AVEC LES LABELS ORIGINAUX")
    print("-" * 60)
    
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    
    cluster_labels = final_result['model'].labels_
    
    if y_train is not None:
        # Métriques de comparaison
        ari = adjusted_rand_score(y_train, cluster_labels)
        nmi = normalized_mutual_info_score(y_train, cluster_labels)
        
        print(f"📊 Comparaison clusters vs labels originaux:")
        print(f"   Adjusted Rand Index: {ari:.4f}")
        print(f"   Normalized Mutual Info: {nmi:.4f}")
        
        # Ajouter aux métriques
        final_result['metrics']['adjusted_rand_index'] = ari
        final_result['metrics']['normalized_mutual_info'] = nmi
        
        # Matrice de confusion clusters vs classes
        confusion = pd.crosstab(pd.Series(cluster_labels, name='Cluster'),
                               pd.Series(y_train, name='Classe réelle'))
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(confusion, annot=True, fmt='d', cmap='YlOrRd')
        plt.title(f'Clusters vs Classes réelles - {method_name}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        confusion_filename = f"clarans_confusion_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
        plt.savefig(confusion_filename, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"💾 Matrice de confusion sauvegardée: {confusion_filename}")
        final_result['plots']['confusion'] = confusion_filename
    
    # Stocker les résultats complets
    method_result = {
        'method': method_name,
        'data_shape': X_train.shape,
        'normalized_shape': X_normalized.shape,
        'best_scoring': best_scoring,
        'final_result': final_result,
        'results_by_metric': results_by_metric
    }
    
    all_results.append(method_result)
    
    # Afficher le résumé
    print("\n📋 RÉSUMÉ DES RÉSULTATS:")
    print("-" * 60)
    metrics = final_result['metrics']
    print(f"Meilleure métrique d'optimisation: {best_scoring}")
    print(f"Nombre de clusters optimaux: {metrics['n_clusters']}")
    print(f"Taille des clusters: {dict(metrics['cluster_sizes'])}")
    print(f"Silhouette Score: {metrics['silhouette']:.4f}")
    print(f"Calinski-Harabasz Index: {metrics['calinski_harabasz']:.2f}")
    print(f"Davies-Bouldin Index: {metrics['davies_bouldin']:.4f}")
    print(f"Temps d'optimisation: {metrics['optimization_time']:.2f}s")
    
    if 'adjusted_rand_index' in metrics:
        print(f"Adjusted Rand Index: {metrics['adjusted_rand_index']:.4f}")
        print(f"Normalized Mutual Info: {metrics['normalized_mutual_info']:.4f}")
    
    print(f"\n🎯 Meilleurs paramètres:")
    for param, value in metrics['best_params'].items():
        print(f"  {param}: {value}")
    
    print(f"\n💾 Fichiers générés:")
    for plot_name, plot_file in final_result['plots'].items():
        print(f"  • {plot_name}: {plot_file}")

# ========================================
# PARTIE 5: COMPARAISON FINALE DES MÉTHODES
# ========================================

if all_results:
    print("\n" + "=" * 80)
    print("📊 COMPARAISON FINALE DES MÉTHODES")
    print("=" * 80)
    
    # Préparer les données pour la comparaison
    comparison_data = []
    
    for result in all_results:
        metrics = result['final_result']['metrics']
        comp_data = {
            'Méthode': result['method'],
            'Échantillons': result['data_shape'][0],
            'Features': result['data_shape'][1],
            'Métrique optimisée': result['best_scoring'],
            'Clusters': metrics['n_clusters'],
            'Silhouette': metrics['silhouette'],
            'Calinski-Harabasz': metrics['calinski_harabasz'],
            'Davies-Bouldin': metrics['davies_bouldin'],
            'Temps optimisation (s)': metrics['optimization_time']
        }
        
        if 'adjusted_rand_index' in metrics:
            comp_data['ARI'] = metrics['adjusted_rand_index']
            comp_data['NMI'] = metrics['normalized_mutual_info']
        
        comparison_data.append(comp_data)
    
    # Créer le DataFrame de comparaison
    comparison_df = pd.DataFrame(comparison_data)
    
    print("\n📋 TABLEAU COMPARATIF:")
    print("-" * 120)
    print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    print("-" * 120)
    
    # Sauvegarder le tableau comparatif
    comparison_filename = f"clarans_bayesian_comparison_{timestamp}.csv"
    comparison_df.to_csv(comparison_filename, index=False)
    print(f"\n💾 Tableau comparatif sauvegardé: {comparison_filename}")
    
    # Graphique de comparaison finale
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    methods = [r['method'] for r in all_results]
    
    # Métriques à comparer
    silhouette_scores = [r['final_result']['metrics']['silhouette'] for r in all_results]
    calinski_scores = [r['final_result']['metrics']['calinski_harabasz'] for r in all_results]
    davies_scores = [r['final_result']['metrics']['davies_bouldin'] for r in all_results]
    n_clusters = [r['final_result']['metrics']['n_clusters'] for r in all_results]
    opt_times = [r['final_result']['metrics']['optimization_time'] for r in all_results]
    
    # Normaliser Calinski-Harabasz pour l'affichage
    calinski_max = max(calinski_scores) if max(calinski_scores) > 0 else 1
    calinski_norm = [c / calinski_max for c in calinski_scores]
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    # Graphique 1: Silhouette Scores
    bars1 = axes[0, 0].bar(methods, silhouette_scores, 
                          color=['green' if s > 0.5 else 'orange' if s > 0.3 else 'red' 
                                for s in silhouette_scores],
                          edgecolor='black')
    axes[0, 0].set_title('Silhouette Scores', fontsize=13, fontweight='bold')
    axes[0, 0].set_ylabel('Score', fontsize=11)
    axes[0, 0].set_ylim([0, 1])
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    for bar in bars1:
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=10)
    
    # Graphique 2: Calinski-Harabasz (normalisé)
    bars2 = axes[0, 1].bar(methods, calinski_norm, 
                          color='skyblue', edgecolor='black')
    axes[0, 1].set_title('Calinski-Harabasz (normalisé)', fontsize=13, fontweight='bold')
    axes[0, 1].set_ylabel('Score normalisé', fontsize=11)
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    for bar, orig in zip(bars2, calinski_scores):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{orig:.0f}', ha='center', va='bottom', fontsize=10)
    
    # Graphique 3: Davies-Bouldin (inversé, plus bas = mieux)
    davies_inverted = [1 / d if d > 0 else 0 for d in davies_scores]
    bars3 = axes[0, 2].bar(methods, davies_inverted, 
                          color=['green' if d > 0.5 else 'orange' if d > 0.3 else 'red' 
                                for d in davies_inverted],
                          edgecolor='black')
    axes[0, 2].set_title('Davies-Bouldin (inversé)', fontsize=13, fontweight='bold')
    axes[0, 2].set_ylabel('1 / Davies-Bouldin', fontsize=11)
    axes[0, 2].set_ylim([0, 1])
    axes[0, 2].grid(True, alpha=0.3, axis='y')
    
    for bar, orig in zip(bars3, davies_scores):
        height = bar.get_height()
        axes[0, 2].text(bar.get_x() + bar.get_width()/2., height,
                       f'{orig:.3f}', ha='center', va='bottom', fontsize=10)
    
    # Graphique 4: Nombre de clusters
    bars4 = axes[1, 0].bar(methods, n_clusters, 
                          color='lightcoral', edgecolor='black')
    axes[1, 0].set_title('Nombre de clusters', fontsize=13, fontweight='bold')
    axes[1, 0].set_ylabel('Nombre', fontsize=11)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    for bar in bars4:
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                       str(int(height)), ha='center', va='bottom', fontsize=10)
    
    # Graphique 5: Temps d'optimisation
    bars5 = axes[1, 1].bar(methods, opt_times, 
                          color='lightblue', edgecolor='black')
    axes[1, 1].set_title('Temps d\'optimisation', fontsize=13, fontweight='bold')
    axes[1, 1].set_ylabel('Secondes', fontsize=11)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    for bar in bars5:
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}s', ha='center', va='bottom', fontsize=10)
    
    # Graphique 6: Score composite
    axes[1, 2].remove()
    ax_composite = fig.add_subplot(2, 3, 6)
    
    # Calculer un score composite
    composite_scores = []
    for i in range(len(methods)):
        sil_norm = silhouette_scores[i]
        cal_norm = calinski_norm[i]
        dav_norm = davies_inverted[i]
        # Poids: Silhouette 40%, Calinski 30%, Davies 30%
        composite = 0.4 * sil_norm + 0.3 * cal_norm + 0.3 * dav_norm
        composite_scores.append(composite)
    
    bars6 = ax_composite.bar(methods, composite_scores, 
                            color=colors, edgecolor='black')
    ax_composite.set_title('Score composite (pondéré)', fontsize=13, fontweight='bold')
    ax_composite.set_ylabel('Score', fontsize=11)
    ax_composite.set_ylim([0, 1])
    ax_composite.grid(True, alpha=0.3, axis='y')
    
    for bar in bars6:
        height = bar.get_height()
        ax_composite.text(bar.get_x() + bar.get_width()/2., height,
                         f'{height:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.suptitle(f'Comparaison des méthodes - CLARANS avec optimisation bayésienne\n{timestamp}', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    final_comparison_plot = f"clarans_bayesian_final_comparison_{timestamp}.png"
    plt.savefig(final_comparison_plot, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"\n💾 Graphique de comparaison finale sauvegardé: {final_comparison_plot}")
    
    # Déterminer la meilleure méthode
    best_idx = np.argmax(composite_scores)
    best_method = all_results[best_idx]
    
    print(f"\n🏆 MEILLEURE MÉTHODE GLOBALE: {best_method['method']}")
    print(f"   Score composite: {composite_scores[best_idx]:.4f}")
    print(f"   Silhouette Score: {silhouette_scores[best_idx]:.4f}")
    print(f"   Nombre de clusters optimaux: {n_clusters[best_idx]}")
    print(f"   Métrique d'optimisation: {best_method['best_scoring']}")
    
    print(f"\n📁 RÉCAPITULATIF DES FICHIERS GÉNÉRÉS:")
    print("-" * 80)
    
    for result in all_results:
        print(f"\n{result['method']}:")
        for plot_name, plot_file in result['final_result']['plots'].items():
            print(f"  • {plot_file}")
    
    print(f"\nFichiers de comparaison:")
    print(f"  • {comparison_filename}")
    print(f"  • {final_comparison_plot}")

else:
    print("❌ Aucune donnée disponible pour l'analyse")

print("\n" + "=" * 80)
print("✅ OPTIMISATION BAYÉSIENNE CLARANS TERMINÉE AVEC SUCCÈS!")
print("=" * 80)